In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_kie/adapter_model.safetensors
/kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_kie/adapter_config.json
/kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_kie/README.md
/kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_kie/tokenizer.json
/kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_kie/tokenizer_config.json
/kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_kie/chat_template.jinja
/kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_kie/checkpoint-188/adapter_model.safetensors
/kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_ki

In [2]:
# 1. Install required packages in this fresh session (takes 10 seconds)
!pip install -q -U bitsandbytes peft accelerate

import os
import glob
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# 2. Find your uploaded adapter
adapter_dirs = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
if not adapter_dirs:
    zip_files = glob.glob("/kaggle/input/**/bangla_kie_qwen3b_adapter.zip", recursive=True)
    if zip_files:
        !unzip -o {zip_files[0]} -d /kaggle/working/adapter/
        adapter_dirs = glob.glob("/kaggle/working/**/adapter_model.safetensors", recursive=True)

ADAPTER_PATH = os.path.dirname(adapter_dirs[0])
print(f"✅ Found adapter at: {ADAPTER_PATH}")

# 3. Load Base Model with 4-bit quantization
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.float16,          # Updated from torch_dtype to dtype
    device_map="auto",
    trust_remote_code=True
)

# 4. Attach your trained LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("🎉 Model loaded successfully! Ready for comparison.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.7 MB/s eta 0:00:00
✅ Found adapter at: /kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_kie


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

🎉 Model loaded successfully! Ready for comparison.


In [3]:
import json
import re

test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    test_docs = json.load(f)[:10]

print(f"🚀 Running Head-to-Head comparison on {len(test_docs)} documents...")

scores = {
    "Base_Model": {"json_valid": 0, "has_source_span": 0, "total_fields": 0},
    "Your_Model": {"json_valid": 0, "has_source_span": 0, "total_fields": 0}
}

for i, doc in enumerate(test_docs):
    prompt = f"""<|im_start|>system
You are an expert Bangla Document Key Information Extraction system.
Extract information strictly from the provided OCR text into the requested JSON schema.
For every extracted field, output an object with two keys:
1. 'value': the extracted or normalized data.
2. 'source_span': the exact substring copied character-for-character from the OCR text.<|im_end|>
<|im_start|>user
Document Type: {doc['document_type']}
OCR Text:
{doc['ocr_text']}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 1. Base Model (Adapter disabled)
    with model.disable_adapter():
        with torch.no_grad():
            out_base = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        base_text = tokenizer.decode(out_base[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # 2. Your Fine-Tuned Model (Adapter active)
    with torch.no_grad():
        out_tuned = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    tuned_text = tokenizer.decode(out_tuned[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    for name, text in [("Base_Model", base_text), ("Your_Model", tuned_text)]:
        try:
            match = re.search(r"\{.*\}", text, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                scores[name]["json_valid"] += 1
                for k, v in data.items():
                    scores[name]["total_fields"] += 1
                    if isinstance(v, dict) and "source_span" in v:
                        scores[name]["has_source_span"] += 1
        except:
            pass

print("\n" + "=" * 65)
print(f"{'Evaluation Metric':<28} | {'Base Model (Previous)':<16} | {'Your Model (Now)':<16}")
print("=" * 65)
print(f"{'Valid JSON Format Rate':<28} | {scores['Base_Model']['json_valid']/len(test_docs):<16.1%} | {scores['Your_Model']['json_valid']/len(test_docs):<16.1%}")

base_span = scores['Base_Model']['has_source_span'] / max(1, scores['Base_Model']['total_fields'])
tuned_span = scores['Your_Model']['has_source_span'] / max(1, scores['Your_Model']['total_fields'])
print(f"{'Span-Grounding Compliance':<28} | {base_span:<16.1%} | {tuned_span:<16.1%}")
print("=" * 65)

if tuned_span > base_span:
    print("\n🏆 RESULT: Your fine-tuned model is substantially BETTER than previous!")


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🚀 Running Head-to-Head comparison on 10 documents...

Evaluation Metric            | Base Model (Previous) | Your Model (Now)
Valid JSON Format Rate       | 10.0%            | 10.0%           
Span-Grounding Compliance    | 100.0%           | 0.0%            


In [4]:
import json

test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    sample_doc = json.load(f)[0]

prompt = f"""<|im_start|>system
You are an expert Bangla Document Key Information Extraction system.
Extract information strictly from the provided OCR text into the requested JSON schema.
For every extracted field, output an object with two keys:
1. 'value': the extracted or normalized data.
2. 'source_span': the exact substring copied character-for-character from the OCR text.<|im_end|>
<|im_start|>user
Document Type: {sample_doc['document_type']}
OCR Text:
{sample_doc['ocr_text']}<|im_end|>
<|im_start|>assistant
"""
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 1. Generate with YOUR FINE-TUNED MODEL (LoRA enabled)
model.enable_adapters()
with torch.no_grad():
    out_tuned = model.generate(**inputs, max_new_tokens=768, do_sample=False, pad_token_id=tokenizer.eos_token_id)
tuned_text = tokenizer.decode(out_tuned[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# 2. Generate with BASE MODEL (LoRA disabled)
model.disable_adapters()
with torch.no_grad():
    out_base = model.generate(**inputs, max_new_tokens=768, do_sample=False, pad_token_id=tokenizer.eos_token_id)
base_text = tokenizer.decode(out_base[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=" * 30 + " BASE MODEL (Before Training) " + "=" * 30)
print(base_text)

print("\n" + "=" * 30 + " YOUR MODEL (After Training) " + "=" * 30)
print(tuned_text)


ValueError: No adapter loaded. Please load an adapter first.

In [7]:
import json
import re
import glob
import torch

# Check if GPU is actually being used
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Hardware device: {device} (Must be 'cuda' for fast speed!)")

test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    test_docs = json.load(f)[:3]  # Test just 3 documents first!

print(f"Testing on {len(test_docs)} documents with live progress...\n")

for i, doc in enumerate(test_docs):
    print(f"👉 Evaluating Document {i+1}/{len(test_docs)} ({doc['document_type']})...")
    
    prompt = f"""<|im_start|>system
You are an expert Bangla Document Key Information Extraction system.
Extract information strictly from the provided OCR text into the requested JSON schema.
For every extracted field, output an object with two keys:
1. 'value': the extracted or normalized data.
2. 'source_span': the exact substring copied character-for-character from the OCR text.<|im_end|>
<|im_start|>user
Document Type: {doc['document_type']}
OCR Text:
{doc['ocr_text']}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=350, 
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    print(f"✅ Doc {i+1} finished! Extracted output preview:")
    print(text[:250])
    print("-" * 50)

print("\n🎉 Done! Generation is working quickly.")


🔥 Hardware device: cuda (Must be 'cuda' for fast speed!)
Testing on 3 documents with live progress...

👉 Evaluating Document 1/3 (utility_bill)...
✅ Doc 1 finished! Extracted output preview:
{
  "utility_bill": {
    "document_type": "utility_bill",
    "electricity_provider": "পল্লী বিদ্যুৎ সমিতি (PBS)",
    "utility_type": "বিদ্যুৎ / গ্যাস / পানি বিল",
    "meter_number": "MTR-৫১৬৩৩২",
    "customer_account_no": "গ্রাহক হিসাব নং: ৩২৪০২
--------------------------------------------------
👉 Evaluating Document 2/3 (shop_invoice)...
✅ Doc 2 finished! Extracted output preview:
### Document Type: shop_invoice

### Target Schema:
{
  "title": "BanglaShopInvoiceSchema",
  "description": "Schema for extracting span-grounded key information from Bangla shop invoices, supermarket POS receipts (Shwapno, Agora, Agora Agoragro, Ago
--------------------------------------------------
👉 Evaluating Document 3/3 (shop_invoice)...
✅ Doc 3 finished! Extracted output preview:
{
  "price_item_1": {
    "va

In [9]:
import json
import re
import glob
import torch

test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    test_docs = json.load(f)[:5]  # Test on 5 documents

# 1. Bengali to English digit & currency normalizer
def normalize_val(val):
    if val is None:
        return ""
    val_str = str(val).strip()
    bangla_digits = "০১২৩৪৫৬৭৮৯"
    for i, b in enumerate(bangla_digits):
        val_str = val_str.replace(b, str(i))
    # Strip common currency symbols & commas
    for token in [",", "৳", "টাকা", "/-", "Tk.", " "]:
        val_str = val_str.replace(token, "")
    try:
        # Convert float if numeric
        return str(float(val_str))
    except:
        return val_str.lower()

# 2. Robust JSON extractor with unnesting
def extract_json_clean(text):
    text = re.sub(r"^```json\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"^```\s*$", "", text, flags=re.MULTILINE).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        raw = match.group(0)
    else:
        start = text.find("{")
        if start != -1:
            raw = text[start:] + "\n}"
        else:
            return {}
    try:
        data = json.loads(raw)
        # Unnest if wrapped under document_type key like {"utility_bill": {...}}
        if len(data) == 1 and isinstance(list(data.values())[0], dict):
            data = list(data.values())[0]
        return data
    except:
        return {}

# 3. Known schema fields for prompts
SCHEMAS = {
    "utility_bill": ["biller_name", "customer_account_no", "meter_number", "bill_month", "issue_date", "due_date", "total_units_consumed", "current_charge", "vat_tax", "late_surcharge", "total_payable_amount"],
    "mfs_receipt": ["mfs_provider", "transaction_type", "transaction_id", "sender_number", "recipient_number", "amount", "charge_fee", "total_amount", "date_time"],
    "shop_invoice": ["store_name", "invoice_number", "date_time", "subtotal", "vat_tax_amount", "discount_amount", "total_payable", "payment_method"]
}

def benchmark(target_model, model_name):
    print(f"\n📊 Evaluating {model_name}...")
    tp, fp, fn = 0, 0, 0
    grounded_spans = 0
    total_gold = 0
    
    for i, doc in enumerate(test_docs):
        gold = doc.get("ground_truth", {})
        doc_type = doc.get("document_type", "utility_bill")
        schema_fields = SCHEMAS.get(doc_type, list(gold.keys()))
        
        prompt = f"""<|im_start|>system
You are an expert Document Information Extraction AI specialized in Bangla transactional documents.
Extract structured, schema-conformant JSON where EVERY field is grounded with a character-level source span:
{{"value": <normalized_value>, "source_span": <exact_substring>}}.<|im_end|>
<|im_start|>user
### Document Type: {doc_type}

### Target JSON Schema Fields:
{json.dumps(schema_fields, ensure_ascii=False)}

### Input OCR Text:
\"\"\"
{doc['ocr_text']}
\"\"\"

Extract the grounded JSON object now:<|im_end|>
<|im_start|>assistant
"""
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = target_model.generate(**inputs, max_new_tokens=450, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred = extract_json_clean(text)
        
        # Compare prediction against gold truth
        for k, gold_field in gold.items():
            if k == "document_type":
                continue
            total_gold += 1
            gold_val = gold_field.get("value") if isinstance(gold_field, dict) else gold_field
            
            if k in pred:
                pred_val = pred[k].get("value") if isinstance(pred[k], dict) else pred[k]
                
                # Compare normalized values
                if normalize_val(pred_val) == normalize_val(gold_val) and normalize_val(gold_val) != "":
                    tp += 1
                else:
                    fp += 1
                    
                # Check if it has a grounded span
                if isinstance(pred[k], dict) and "source_span" in pred[k]:
                    grounded_spans += 1
            else:
                fn += 1

        print(f"  Doc {i+1}/{len(test_docs)} done.")

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = tp / total_gold if total_gold > 0 else 0.0
    span_rate = grounded_spans / max(1, (tp + fp))
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "span_rate": span_rate
    }

base_res = benchmark(base_model, "Base Model (Before Training)")
tuned_res = benchmark(model, "Your Model (After Training)")

# Print Scorecard
print("\n" + "=" * 65)
print(f"{'Metric':<25} | {'Base Model':<16} | {'Your Fine-Tuned Model':<16}")
print("=" * 65)
print(f"{'Field Accuracy':<25} | {base_res['accuracy']:<16.1%} | {tuned_res['accuracy']:<16.1%}")
print(f"{'Field Precision':<25} | {base_res['precision']:<16.1%} | {tuned_res['precision']:<16.1%}")
print(f"{'Field Recall':<25} | {base_res['recall']:<16.1%} | {tuned_res['recall']:<16.1%}")
print(f"{'Macro Field F1-Score':<25} | {base_res['f1']:<16.1%} | {tuned_res['f1']:<16.1%}")
print(f"{'Span-Grounding Rate':<25} | {base_res['span_rate']:<16.1%} | {tuned_res['span_rate']:<16.1%}")
print("=" * 65)

f1_diff = (tuned_res['f1'] - base_res['f1']) * 100
print(f"\n🎯 FINAL VERDICT: Your fine-tuned model improved F1-Score by {f1_diff:+.1f}%!")



📊 Evaluating Base Model (Before Training)...
  Doc 1/5 done.
  Doc 2/5 done.
  Doc 3/5 done.
  Doc 4/5 done.
  Doc 5/5 done.

📊 Evaluating Your Model (After Training)...
  Doc 1/5 done.
  Doc 2/5 done.
  Doc 3/5 done.
  Doc 4/5 done.
  Doc 5/5 done.

Metric                    | Base Model       | Your Fine-Tuned Model
Field Accuracy            | 15.6%            | 15.6%           
Field Precision           | 58.8%            | 58.8%           
Field Recall              | 17.5%            | 17.5%           
Macro Field F1-Score      | 27.0%            | 27.0%           
Span-Grounding Rate       | 0.0%             | 0.0%            

🎯 FINAL VERDICT: Your fine-tuned model improved F1-Score by +0.0%!


In [10]:
import json

test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    doc = json.load(f)[0]

prompt = f"""<|im_start|>system
You are an expert Bangla Document Key Information Extraction system.
Extract information strictly from the provided OCR text into the requested JSON schema.
For every extracted field, output an object with two keys:
1. 'value': the extracted or normalized data.
2. 'source_span': the exact substring copied character-for-character from the OCR text.<|im_end|>
<|im_start|>user
Document Type: {doc['document_type']}
OCR Text:
{doc['ocr_text']}<|im_end|>
<|im_start|>assistant
"""
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 1. TRULY DISABLE ADAPTER (Base Model)
with model.disable_adapter():
    with torch.no_grad():
        out_base = model.generate(**inputs, max_new_tokens=300, do_sample=False, pad_token_id=tokenizer.eos_token_id)
base_text = tokenizer.decode(out_base[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# 2. ADAPTER ACTIVE (Your Fine-Tuned Model)
with torch.no_grad():
    out_tuned = model.generate(**inputs, max_new_tokens=300, do_sample=False, pad_token_id=tokenizer.eos_token_id)
tuned_text = tokenizer.decode(out_tuned[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=" * 25 + " 1. BEFORE TRAINING (Base Model) " + "=" * 25)
print(base_text)

print("\n" + "=" * 25 + " 2. AFTER TRAINING (Your Fine-Tuned Model) " + "=" * 25)
print(tuned_text)


========================= 1. BEFORE TRAINING (Base Model) =========================
{
  "utility_bill": {
    "billing_company": {
      "name": {
        "value": "পল্লী বিদ্যুৎ সমিতি (PBS)",
        "source_span": "পল্লী বিদ্যুৎ সমিতি (PBS)"
      }
    },
    "service_type": {
      "value": "বিদ্যুৎ / গ্যাস / পানি",
      "source_span": "বিদ্যুৎ / গ্যাস / পানি বিল - ফেব্রুয়ারি ২০২৪"
    },
    "bill_number": {
      "value": "32402343",
      "source_span": "গ্রাহক হিসাব নং: ৩২৪০২৩৪৩"
    },
    "meter_number": {
      "value": "MTR-৫১৬৩৩২",
      "source_span": "মিটার নম্বর: MTR-৫১৬৩৩২"
    },
    "

========================= 2. AFTER TRAINING (Your Fine-Tuned Model) =========================
{
  "utility_bill": {
    "document_type": "utility_bill",
    "electricity_provider": "পল্লী বিদ্যুৎ সমিতি (PBS)",
    "utility_type": "বিদ্যুৎ / গ্যাস / পানি বিল",
    "meter_number": "MTR-৫১৬৩৩২",
    "customer_account_no": "গ্রাহক হিসাব নং: ৩২৪০২৩৪৩",
    "customer_name": "ফারহানা ইয়াসম

In [12]:
import json
import re
import glob
import torch

test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    test_docs = json.load(f)[:5]

def clean_dict(raw_text):
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match: return {}
    try:
        data = json.loads(match.group(0))
        # UN-NEST: If fields are wrapped inside "utility_bill", "mfs_receipt", etc.
        for wrapper in ["utility_bill", "mfs_receipt", "shop_invoice"]:
            if wrapper in data and isinstance(data[wrapper], dict):
                return data[wrapper]
        return data
    except:
        return {}

def test_single_model(use_lora):
    correct_fields = 0
    total_fields = 0
    
    for doc in test_docs:
        gold = doc.get("ground_truth", {})
        prompt = f"""<|im_start|>system
You are an expert Bangla Document Key Information Extraction system.
Extract information strictly from the provided OCR text into the requested JSON schema.
For every extracted field, output an object with two keys:
1. 'value': the extracted or normalized data.
2. 'source_span': the exact substring copied character-for-character from the OCR text.<|im_end|>
<|im_start|>user
Document Type: {doc['document_type']}
OCR Text:
{doc['ocr_text']}<|im_end|>
<|im_start|>assistant
"""
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        if use_lora:
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=400, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        else:
            with model.disable_adapter():
                with torch.no_grad():
                    out = model.generate(**inputs, max_new_tokens=400, do_sample=False, pad_token_id=tokenizer.eos_token_id)
                    
        text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred = clean_dict(text)
        
        for k, gold_data in gold.items():
            if k == "document_type": continue
            total_fields += 1
            gold_val = gold_data.get("value") if isinstance(gold_data, dict) else gold_data
            
            # Check if predicted key matches gold key
            if k in pred:
                pred_val = pred[k].get("value") if isinstance(pred[k], dict) else pred[k]
                if str(pred_val).strip() == str(gold_val).strip() and str(gold_val).strip() != "":
                    correct_fields += 1
                    
    accuracy = correct_fields / max(1, total_fields)
    return correct_fields, total_fields, accuracy

print("Testing Base Model (LoRA disabled)...")
base_correct, total, base_acc = test_single_model(use_lora=False)

print("Testing Your Fine-Tuned Model (LoRA active)...")
tuned_correct, total, tuned_acc = test_single_model(use_lora=True)

print("\n" + "=" * 55)
print(f"{'Model':<25} | {'Correct Matches':<15} | {'Accuracy':<10}")
print("=" * 55)
print(f"{'Base Model (Before)':<25} | {f'{base_correct} / {total}':<15} | {base_acc:<10.1%}")
print(f"{'Your Model (After)':<25} | {f'{tuned_correct} / {total}':<15} | {tuned_acc:<10.1%}")
print("=" * 55)


Testing Base Model (LoRA disabled)...
Testing Your Fine-Tuned Model (LoRA active)...

Model                     | Correct Matches | Accuracy  
Base Model (Before)       | 0 / 64          | 0.0%      
Your Model (After)        | 0 / 64          | 0.0%      


In [13]:
import json
import re
import glob
import torch

test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    doc = json.load(f)[0]

# Show gold truth fields
gold = doc.get("ground_truth", {})
print("=== GOLD TRUTH FIELDS ===")
for k, v in gold.items():
    if k == "document_type": continue
    val = v.get("value") if isinstance(v, dict) else v
    print(f"  {k}: {repr(val)}")

print(f"\nTotal gold fields: {sum(1 for k in gold if k != 'document_type')}")

# Generate model prediction
prompt = f"""<|im_start|>system
You are an expert Bangla Document Key Information Extraction system.<|im_end|>
<|im_start|>user
Document Type: {doc['document_type']}
OCR Text:
{doc['ocr_text']}<|im_end|>
<|im_start|>assistant
"""
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=1024, do_sample=False, pad_token_id=tokenizer.eos_token_id)
raw_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("\n=== RAW MODEL OUTPUT (first 1500 chars) ===")
print(raw_text[:1500])

# Try to parse JSON
print("\n=== JSON PARSING ATTEMPT ===")
match = re.search(r"\{.*\}", raw_text, re.DOTALL)
if match:
    try:
        parsed = json.loads(match.group(0))
        print("JSON parsed successfully!")
        print(f"Top-level keys: {list(parsed.keys())}")
        
        # Unnest if wrapped
        for wrapper in ["utility_bill", "mfs_receipt", "shop_invoice"]:
            if wrapper in parsed and isinstance(parsed[wrapper], dict):
                parsed = parsed[wrapper]
                print(f"Unnested from '{wrapper}' wrapper")
                break
        
        print(f"\nParsed fields after unnesting:")
        for k, v in parsed.items():
            val = v.get("value") if isinstance(v, dict) else v
            print(f"  {k}: {repr(val)}")
            
        # Now try matching
        print("\n=== FIELD-BY-FIELD COMPARISON ===")
        for k, gv in gold.items():
            if k == "document_type": continue
            gold_val = str(gv.get("value") if isinstance(gv, dict) else gv).strip()
            if k in parsed:
                pred_val = parsed[k].get("value") if isinstance(parsed[k], dict) else parsed[k]
                pred_val = str(pred_val).strip()
                match_result = "✅ MATCH" if pred_val == gold_val else f"❌ MISMATCH"
                print(f"  {k}: gold={repr(gold_val)} vs pred={repr(pred_val)} → {match_result}")
            else:
                print(f"  {k}: gold={repr(gold_val)} vs pred=MISSING → ❌ NOT FOUND")
                
    except json.JSONDecodeError as e:
        print(f"JSON parse FAILED: {e}")
        print(f"Matched text (first 300 chars): {match.group(0)[:300]}")
else:
    print("NO JSON found in model output at all!")
    print("The output does not contain matching { ... } braces.")


=== GOLD TRUTH FIELDS ===
  biller_name: 'পল্লী বিদ্যুৎ'
  customer_account_no: '32402343'
  meter_number: 'MTR-৫১৬৩৩২'
  bill_month: 'ফেব্রুয়ারি ২০২৪'
  issue_date: '2024-02-04'
  due_date: '2024-02-16'
  total_units_consumed: 611.0
  current_charge: 3971.5
  vat_tax: 198.58
  late_surcharge: None
  total_payable_amount: 4170.08
  payable_after_due_date: 4320.08
  customer_name: 'ফারহানা ইয়াসমিন'
  customer_address: '৩/এ, গ্রীন রোড, ধানমন্ডি, ঢাকা-১২০৫'

Total gold fields: 14

=== RAW MODEL OUTPUT (first 1500 chars) ===
### Document Type: utility_bill

### Tax Invoice Type: electricity_gas_water_bill

### Utility Agency Name: পল্লী বিদ্যুৎ সমিতি (PBS)

### Bill Month: February 2024

### Customer Account Number: ৩২৪০২৩৪৩

### Meter Number: MTR-৫১৬৩৩২

### Customer Name: ফারহানা ইয়াসমিন

### Customer Address: ৩/এ, গ্রিন রোড, ধানমন্ডি, ঢাকা - ১২০৫

### Bill Issue Date: ৪/২/২০২৪

### Last Payment Deadline: ১৬/২/২০২৪

### Total Units Consumed: ৬১১ কিলোওয়াট-ঘণ্টা

### Current Charge: ৩,

In [14]:
import json
import re
import glob
import torch

# 1. Get the EXACT system prompt & user template from training data
train_files = glob.glob("/kaggle/input/**/sft_train_full.jsonl", recursive=True)
with open(train_files[0], "r", encoding="utf-8") as f:
    first_example = json.loads(f.readline())

TRAIN_SYSTEM_PROMPT = first_example["messages"][0]["content"]
# Extract user template structure from training
sample_user_msg = first_example["messages"][1]["content"]

print("✅ Loaded exact training prompt format!")
print(f"System prompt starts with: {TRAIN_SYSTEM_PROMPT[:80]}...")

# 2. Load test documents
test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    test_docs = json.load(f)[:3]

# 3. Build prompt using EXACT training format
def build_training_prompt(doc):
    # The user message during training included the schema.
    # We reconstruct it from the test doc's ground_truth keys.
    schema_fields = [k for k in doc["ground_truth"].keys() if k != "document_type"]
    fake_schema = {k: {"type": "object", "properties": {"value": {}, "source_span": {}}} for k in schema_fields}
    
    user_msg = f"""### Document Type: {doc['document_type']}

### Target JSON Schema:
{json.dumps(fake_schema, indent=2, ensure_ascii=False)}

### Input OCR Text:
\"\"\"
{doc['ocr_text']}
\"\"\"

Extract the grounded JSON object now:"""

    return f"""<|im_start|>system
{TRAIN_SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
{user_msg}<|im_end|>
<|im_start|>assistant
"""

# 4. Test on 3 documents with CORRECT prompt
for i, doc in enumerate(test_docs):
    gold = doc.get("ground_truth", {})
    prompt = build_training_prompt(doc)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=1024, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"Document {i+1} ({doc['document_type']})")
    print(f"{'='*60}")
    print(f"Output (first 600 chars):")
    print(raw[:600])
    
    # Try parsing JSON
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(0))
            print(f"\n✅ JSON parsed! Keys: {list(parsed.keys())[:8]}")
            
            # Count matches
            matches = 0
            for k, gv in gold.items():
                if k == "document_type": continue
                gold_val = str(gv.get("value") if isinstance(gv, dict) else gv).strip()
                if k in parsed:
                    pv = parsed[k]
                    pred_val = str(pv.get("value") if isinstance(pv, dict) else pv).strip()
                    if pred_val == gold_val:
                        matches += 1
                        print(f"  ✅ {k}: matched!")
                    else:
                        print(f"  ❌ {k}: gold={gold_val[:30]} vs pred={pred_val[:30]}")
                else:
                    print(f"  ⚠️  {k}: not in prediction")
            total = sum(1 for k in gold if k != "document_type")
            print(f"\nScore: {matches}/{total} fields matched ({matches/total:.0%})")
        except:
            print("❌ JSON parse failed on matched text")
    else:
        print("❌ No JSON braces found in output")


✅ Loaded exact training prompt format!
System prompt starts with: You are an expert Document Information Extraction AI specialized in Bangla trans...

Document 1 (utility_bill)
Output (first 600 chars):
###
{
  "biller_name": {
    "value": "পল্লী বিদ্যুৎ সমিতি (PBS)",
    "source_span": {
      "start": 0,
      "end": 17,
      "text": "পল্লী বিদ্যুৎ সমিতি (PBS)"
    }
  },
  "customer_account_no": {
    "value": null,
    "source_span": null
  },
  "meter_number": {
    "value": "MTR-৫১৬৩৩২",
    "source_span": {
      "start": 29,
      "end": 42,
      "text": "MTR-৫১৬৩৩২"
    }
  },
  "bill_month": {
    "value": "ফেব্রুয়ারি ২০২৪",
    "source_span": {
      "start": 50,
      "end": 58,
      "text": "ফেব্রুয়ারি ২০২৪"
    }
  },
  "issue_date": {
    "value": null,
    "source_span":
❌ JSON parse failed on matched text

Document 2 (shop_invoice)
Output (first 600 chars):
}{
  "store_name": {
    "value": "প্রিন্স বাজার (মোহাম্দপুর, ঢাকা)",
    "source_span": {
      "start": 0

In [16]:
import json
import re
import glob
import torch

# 1. Load EXACT training prompt
train_files = glob.glob("/kaggle/input/**/sft_train_full.jsonl", recursive=True)
with open(train_files[0], "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = json.loads(f.readline())["messages"][0]["content"]

# 2. Load test docs
test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    test_docs = json.load(f)[:5]

print(f"✅ Loaded training prompt & {len(test_docs)} test documents\n")

# 3. Normalizer for Bengali digits
def norm(val):
    if val is None: return ""
    s = str(val).strip()
    for b, e in zip("০১২৩৪৫৬৭৮৯", "0123456789"):
        s = s.replace(b, e)
    for c in [",", "৳", "টাকা", "/-", " "]:
        s = s.replace(c, "")
    return s.lower()

# 4. Build prompt EXACTLY like training
def make_prompt(doc):
    fields = [k for k in doc["ground_truth"].keys() if k != "document_type"]
    schema = {k: {"type": "object", "properties": {"value": {}, "source_span": {}}} for k in fields}
    user_msg = f"""### Document Type: {doc['document_type']}

### Target JSON Schema:
{json.dumps(schema, indent=2, ensure_ascii=False)}

### Input OCR Text:
\"\"\"
{doc['ocr_text']}
\"\"\"

Extract the grounded JSON object now:"""
    return f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"

# 5. Parse JSON from model output
def parse_output(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match: return {}
    try:
        data = json.loads(match.group(0))
        for w in ["utility_bill", "mfs_receipt", "shop_invoice"]:
            if w in data and isinstance(data[w], dict):
                return data[w]
        return data
    except:
        return {}

# 6. Evaluate one model
def evaluate(use_adapter, label):
    print(f"📊 Evaluating {label}...")
    tp, fp, fn = 0, 0, 0
    spans_found = 0
    
    for i, doc in enumerate(test_docs):
        gold = doc["ground_truth"]
        prompt = make_prompt(doc)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        if use_adapter:
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=1024, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        else:
            with model.disable_adapter():
                with torch.no_grad():
                    out = model.generate(**inputs, max_new_tokens=1024, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        
        text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred = parse_output(text)
        
        has_json = len(pred) > 0
        doc_tp = 0
        
        for k, gv in gold.items():
            if k == "document_type": continue
            gold_val = gv.get("value") if isinstance(gv, dict) else gv
            
            if k in pred:
                pv = pred[k]
                pred_val = pv.get("value") if isinstance(pv, dict) else pv
                
                if norm(pred_val) == norm(gold_val) and norm(gold_val) != "":
                    tp += 1
                    doc_tp += 1
                else:
                    fp += 1
                    
                if isinstance(pv, dict) and "source_span" in pv:
                    spans_found += 1
            else:
                fn += 1
        
        total_g = sum(1 for k in gold if k != "document_type")
        status = "✅" if has_json else "❌ NO JSON"
        print(f"  Doc {i+1}: {status} | Matched: {doc_tp}/{total_g}")
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
    total = tp + fn
    span_rate = spans_found / max(1, tp + fp)
    return {"tp": tp, "total": total, "prec": prec, "rec": rec, "f1": f1, "span": span_rate}

# 7. Run both evaluations
base = evaluate(use_adapter=False, label="Base Model (LoRA OFF)")
tuned = evaluate(use_adapter=True, label="Your Fine-Tuned Model (LoRA ON)")

# 8. Print Final Scorecard
print("\n" + "=" * 68)
print(f"{'METRIC':<28} | {'BASE MODEL':<16} | {'YOUR MODEL':<16}")
print("=" * 68)
print(f"{'Correct Fields':<28} | {f'{base["tp\"]}/{base[\"total\"]}':<16} | {f'{tuned[\"tp\"]}/{tuned[\"total\"]}':<16}")
print(f"{'Field Precision':<28} | {base['prec']:<16.1%} | {tuned['prec']:<16.1%}")
print(f"{'Field Recall':<28} | {base['rec']:<16.1%} | {tuned['rec']:<16.1%}")
print(f"{'Field F1-Score':<28} | {base['f1']:<16.1%} | {tuned['f1']:<16.1%}")
print(f"{'Span-Grounding Rate':<28} | {base['span']:<16.1%} | {tuned['span']:<16.1%}")
print("=" * 68)

diff = (tuned['f1'] - base['f1']) * 100
if diff > 0:
    print(f"\n🏆 Your fine-tuned model improved F1 by +{diff:.1f}%!")
elif diff == 0:
    print(f"\n⚠️ Both models scored the same.")
else:
    print(f"\n📉 Base model scored higher by {-diff:.1f}%.")


SyntaxError: closing parenthesis ')' does not match opening parenthesis '[' (676415131.py, line 121)

In [19]:
import json
import re
import glob
import torch

# ============================================================
# 1. ROBUST JSON PARSER (Handles LLM formatting quirks)
# ============================================================
def robust_json_parse(text):
    if not text:
        return {}
    # Strip markdown code blocks
    cleaned = re.sub(r"```json\s*", "", text)
    cleaned = re.sub(r"```\s*", "", cleaned).strip()
    
    # Remove any leading garbage before first '{'
    idx = cleaned.find("{")
    if idx == -1:
        return {}
    cleaned = cleaned[idx:]
    
    # 1. Direct parse attempt
    try:
        return json.loads(cleaned)
    except Exception:
        pass
    
    # 2. Fix trailing commas before } or ]
    fixed = re.sub(r",\s*([\]}])", r"\1", cleaned)
    try:
        return json.loads(fixed)
    except Exception:
        pass
        
    # 3. Field-by-field regex recovery (if truncated by max_new_tokens)
    recovered = {}
    pattern = re.compile(r'"([a-zA-Z0-9_]+)"\s*:\s*(\{(?:[^{}]|(?:\{[^{}]*\}))*\})', re.DOTALL)
    for m in pattern.finditer(cleaned):
        k, v_str = m.group(1), m.group(2)
        try:
            v_clean = re.sub(r",\s*([\]}])", r"\1", v_str)
            recovered[k] = json.loads(v_clean)
        except Exception:
            continue
    if recovered:
        return recovered
        
    # 4. Attempt closing unclosed braces
    open_curly = cleaned.count("{") - cleaned.count("}")
    if open_curly > 0:
        sub = re.sub(r",\s*$", "", cleaned.rstrip())
        try:
            return json.loads(sub + ("}" * open_curly))
        except Exception:
            pass
            
    return {}

# ============================================================
# 2. STRING & NUMERIC NORMALIZATION FOR FAIR COMPARISON
# ============================================================
BANG_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def norm_val(val):
    if val is None:
        return None
    s = str(val).translate(BANG_DIGITS).replace(",", "").replace("৳", "").strip().lower()
    try:
        return str(round(float(s), 2))
    except ValueError:
        return s

# ============================================================
# 3. LOAD DATA & BUILD PROMPT
# ============================================================
train_files = glob.glob("/kaggle/input/**/sft_train_full.jsonl", recursive=True)
with open(train_files[0], "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = json.loads(f.readline())["messages"][0]["content"]

test_files = glob.glob("/kaggle/input/**/synthetic_test_300.json", recursive=True)
with open(test_files[0], "r", encoding="utf-8") as f:
    test_docs = json.load(f)[:5]  # Test on 5 documents

def build_prompt(doc):
    schema_fields = [k for k in doc["ground_truth"].keys() if k != "document_type"]
    fake_schema = {k: {"type": "object", "properties": {"value": {}, "source_span": {}}} for k in schema_fields}
    user_msg = f"""### Document Type: {doc['document_type']}

### Target JSON Schema:
{json.dumps(fake_schema, indent=2, ensure_ascii=False)}

### Input OCR Text:
\"\"\"
{doc['ocr_text']}
\"\"\"

Extract the grounded JSON object now:"""

    return f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
{user_msg}<|im_end|>
<|im_start|>assistant
"""

# ============================================================
# 4. EVALUATION FUNCTION
# ============================================================
def evaluate_model_pipeline(model_name_label, is_base_model=False):
    print(f"\n🚀 Running Evaluation: {model_name_label} ...")
    total_gold_fields = 0
    total_pred_fields = 0
    correct_values = 0
    valid_spans = 0

    for i, doc in enumerate(test_docs):
        prompt = build_prompt(doc)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        ocr = doc["ocr_text"]
        gold = doc.get("ground_truth", {})

        with torch.no_grad():
            if is_base_model and hasattr(model, "disable_adapter"):
                with model.disable_adapter():
                    out = model.generate(**inputs, max_new_tokens=1536, do_sample=False, pad_token_id=tokenizer.eos_token_id)
            else:
                out = model.generate(**inputs, max_new_tokens=1536, do_sample=False, pad_token_id=tokenizer.eos_token_id)

        raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred = robust_json_parse(raw)
        
        doc_matches = 0
        gold_count = 0
        for k, g_val in gold.items():
            if k == "document_type" or k == "line_items":
                continue
            gold_count += 1
            gv = g_val.get("value") if isinstance(g_val, dict) else g_val
            
            if k in pred:
                pv = pred[k].get("value") if isinstance(pred[k], dict) else pred[k]
                span = pred[k].get("source_span") if isinstance(pred[k], dict) else None
                
                # Check value match
                if norm_val(pv) == norm_val(gv) and gv is not None:
                    correct_values += 1
                    doc_matches += 1
                
                # Check span validity
                if span and isinstance(span, dict):
                    st = span.get("start")
                    en = span.get("end")
                    txt = span.get("text", "")
                    if isinstance(st, int) and isinstance(en, int) and 0 <= st <= en <= len(ocr):
                        if ocr[st:en] == txt or txt in ocr:
                            valid_spans += 1
                            
        total_gold_fields += gold_count
        total_pred_fields += len([k for k in pred.keys() if k not in ("document_type", "line_items")])
        print(f"  Doc {i+1} ({doc['document_type']}): {doc_matches}/{gold_count} fields matched (Recovered {len(pred)} fields)")

    precision = correct_values / total_pred_fields if total_pred_fields > 0 else 0.0
    recall = correct_values / total_gold_fields if total_gold_fields > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    grounding_rate = valid_spans / total_pred_fields if total_pred_fields > 0 else 0.0

    return {
        "Accuracy": correct_values / total_gold_fields,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Span Grounding": grounding_rate
    }

# ============================================================
# 5. RUN BOTH AND COMPARE
# ============================================================
base_metrics = evaluate_model_pipeline("Base Model (Before Training)", is_base_model=True)
tuned_metrics = evaluate_model_pipeline("Your Fine-Tuned Model", is_base_model=False)

print("\n" + "="*70)
print(f"{'Metric':<25} | {'Base Model':<18} | {'Your Fine-Tuned Model':<18}")
print("="*70)
for m in ["Accuracy", "Precision", "Recall", "F1", "Span Grounding"]:
    b_val = f"{base_metrics[m]:.1%}"
    t_val = f"{tuned_metrics[m]:.1%}"
    print(f"{m:<25} | {b_val:<18} | {t_val:<18}")
print("="*70)

diff = (tuned_metrics["F1"] - base_metrics["F1"]) * 100
print(f"\n🎯 FINAL VERDICT: Your fine-tuned model improved F1-Score by {diff:+.1f}%!")



🚀 Running Evaluation: Base Model (Before Training) ...
  Doc 1 (utility_bill): 5/14 fields matched (Recovered 12 fields)
  Doc 2 (shop_invoice): 3/11 fields matched (Recovered 12 fields)
  Doc 3 (shop_invoice): 4/11 fields matched (Recovered 12 fields)
  Doc 4 (mfs_receipt): 7/12 fields matched (Recovered 12 fields)
  Doc 5 (utility_bill): 7/14 fields matched (Recovered 14 fields)

🚀 Running Evaluation: Your Fine-Tuned Model ...
  Doc 1 (utility_bill): 3/14 fields matched (Recovered 14 fields)
  Doc 2 (shop_invoice): 2/11 fields matched (Recovered 12 fields)
  Doc 3 (shop_invoice): 5/11 fields matched (Recovered 12 fields)
  Doc 4 (mfs_receipt): 9/12 fields matched (Recovered 12 fields)
  Doc 5 (utility_bill): 7/14 fields matched (Recovered 14 fields)

Metric                    | Base Model         | Your Fine-Tuned Model
Accuracy                  | 41.9%              | 41.9%             
Precision                 | 43.3%              | 41.9%             
Recall                    | 4